In [1]:
# Standard library
import json
import random
import time
from argparse import ArgumentParser

# Third-party
import numpy as np
import pytorch_lightning as pl
import torch
from lightning_fabric.utilities import seed
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.profilers import AdvancedProfiler

# First-party
from neural_lam import constants, utils, config
from neural_lam.weather_dataset import WeatherDataset
from neural_lam.downscaling_dataset import DownscalingDataset
from neural_lam.netCDF_dataset import NetCDFDataset
from neural_lam.models.graph_efm import GraphEFM
from neural_lam.models.graph_fm import GraphFM
from neural_lam.models.graphcast import GraphCast
from neural_lam.models.diffusion import Diffusion
from neural_lam.models.ir_sde import IR_SDE
from neural_lam.models.stochastic_interpolants import SI

In [2]:
import os

In [3]:
[nc_path for nc_path in os.listdir('/mimer/NOBACKUP/groups/mlhighres/projects/HCLIM-emulator/HCLIM') if '.nc' in nc_path]

['standardized.global.evspsbl_remapped_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951_2014.nc',
 'standardized.global.tasmax_remapped_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951_2014.nc',
 'standardized.global.vas_remapped_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951_2014.nc',
 'standardized.global.huss_remapped_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951_2014.nc',
 'standardized.global.psl_remapped_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951_2014.nc',
 'standardized.global.clt_remapped_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951_2014.nc',
 'standardized.global.hurs_remapped_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLIMcom-SMHI_HCLIM43-ALADIN_v1-r1_day_1951_2014.nc',
 'standardized.global.rsds_remapped_EUR-12_EC-Earth3-Veg_historical_r1i1p1f1_HCLI

In [2]:
MODELS = {
    "graphcast": GraphCast,
    "graph_fm": GraphFM,
    "graph_efm": GraphEFM,
    "diffusion": Diffusion,
    "ir_sde": IR_SDE, 
    "SI": SI,
}

In [16]:
config_loader = config.Config.from_file('/mimer/NOBACKUP/groups/mlhighres/users/mikhaili/neural-lam/neural_lam/clim_config_emulator.yaml')

In [17]:
train_loader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=config_loader.dataset.train_start_date,
        end_date=config_loader.dataset.train_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
        upscale_inputs=config_loader.dataset.upscale_inputs,
        static_fields_files=config_loader.dataset.static_fields_files,
        interpolation_mode=config_loader.dataset.interpolation_mode,
        coordinate_names=config_loader.dataset.coordinate_names,
        provide_coordinates=config_loader.dataset.provide_coordinates,
        provide_day_of_year=config_loader.dataset.provide_day_of_year
    ),
    batch_size=4,
    shuffle=True,
    num_workers=1,
)

input_files: ['standardized.global.tas_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.tasmin_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.tasmax_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.sfcWind_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.uas_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.vas_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.hurs_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.huss_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.psl_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.clt_EC-Earth3-Veg_historical_r1i1p1f1_day_1950-2015_NEA_r360x180.nc', 'standardized.global.rsds_EC-Earth3-Veg_historical_r1i1p1f1_day_

In [18]:
test_batch = None
for batch in train_loader:
    test_batch = batch
    break

In [19]:
test_batch['LQ'].shape

torch.Size([4, 77, 400, 550])

In [20]:
test_batch['LQ'].isnan().any()

tensor(False)

In [21]:
mask = torch.isnan(test_batch['LQ'])

# Reduce over batch, lat, lon → keep only variable dimension
var_has_nan = mask.any(dim=(0, 2, 3))

# Get indices of variables with NaNs
var_indices = torch.nonzero(var_has_nan, as_tuple=True)[0]

print(var_indices)

tensor([], dtype=torch.int64)


In [22]:
test_batch['HQ'].shape

torch.Size([4, 13, 400, 550])

In [23]:
test_batch['HQ'].isnan().any()

tensor(False)

In [22]:
config_loader = config.Config.from_file('neural_lam/ALPS_hist_global_std_config.yaml')

In [23]:
train_loader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=config_loader.dataset.train_start_date,
        end_date=config_loader.dataset.train_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
        upscale_inputs=config_loader.dataset.upscale_inputs,
        static_fields_files=config_loader.dataset.static_fields_files,
        interpolation_mode=config_loader.dataset.interpolation_mode,
        coordinate_names=config_loader.dataset.coordinate_names,
        provide_coordinates=config_loader.dataset.provide_coordinates,
        provide_day_of_year=config_loader.dataset.provide_day_of_year
    ),
    batch_size=4,
    shuffle=True,
    num_workers=1,
)

input_files: ['standardized.q_500_CNRM-CM5_1961-1980.nc', 'standardized.q_700_CNRM-CM5_1961-1980.nc', 'standardized.q_850_CNRM-CM5_1961-1980.nc', 'standardized.t_500_CNRM-CM5_1961-1980.nc', 'standardized.t_700_CNRM-CM5_1961-1980.nc', 'standardized.t_850_CNRM-CM5_1961-1980.nc', 'standardized.u_500_CNRM-CM5_1961-1980.nc', 'standardized.u_700_CNRM-CM5_1961-1980.nc', 'standardized.u_850_CNRM-CM5_1961-1980.nc', 'standardized.v_500_CNRM-CM5_1961-1980.nc', 'standardized.v_700_CNRM-CM5_1961-1980.nc', 'standardized.v_850_CNRM-CM5_1961-1980.nc', 'standardized.z_500_CNRM-CM5_1961-1980.nc', 'standardized.z_700_CNRM-CM5_1961-1980.nc', 'standardized.z_850_CNRM-CM5_1961-1980.nc']
Reading static fields from ['standardized.orog_ALPS.nc']...
Providing ground truth coordinate grid...
Providing day of year encodings...


In [24]:
test_batch = None
for batch in train_loader:
    test_batch = batch
    break

In [25]:
test_batch['LQ'].shape

torch.Size([4, 20, 128, 128])

In [26]:
test_batch['LQ'].isnan().any()

tensor(False)

In [27]:
test_batch['HQ'].shape

torch.Size([4, 2, 128, 128])

In [28]:
test_batch['HQ'].isnan().any()

tensor(False)

In [29]:
config_loader = config.Config.from_file('neural_lam/ALPS_hist_future_global_std_config.yaml')

In [30]:
train_loader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=config_loader.dataset.train_start_date,
        end_date=config_loader.dataset.train_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
        upscale_inputs=config_loader.dataset.upscale_inputs,
        static_fields_files=config_loader.dataset.static_fields_files,
        interpolation_mode=config_loader.dataset.interpolation_mode,
        coordinate_names=config_loader.dataset.coordinate_names,
        provide_coordinates=config_loader.dataset.provide_coordinates,
        provide_day_of_year=config_loader.dataset.provide_day_of_year
    ),
    batch_size=4,
    shuffle=True,
    num_workers=1,
)

input_files: ['standardized.q_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.q_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.q_850_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.t_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.t_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.t_850_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.u_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.u_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.u_850_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.v_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.v_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.v_850_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.z_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.z_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.z_850_CNRM-CM5_1961-1980_2080-2099.nc']
Reading static fields from ['standardized.orog_ALPS.nc']...
Providing ground truth coordinate grid...
Providing day of year encodings...


In [31]:
test_batch = None
for batch in train_loader:
    test_batch = batch
    break

In [32]:
test_batch['LQ'].shape

torch.Size([4, 20, 128, 128])

In [33]:
test_batch['LQ'].isnan().any()

tensor(False)

In [34]:
test_batch['HQ'].shape

torch.Size([4, 2, 128, 128])

In [35]:
test_batch['HQ'].isnan().any()

tensor(False)

In [36]:
config_loader = config.Config.from_file('neural_lam/ALPS_hist_future_local_std_config.yaml')

In [38]:
train_loader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=config_loader.dataset.train_start_date,
        end_date=config_loader.dataset.train_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
        upscale_inputs=config_loader.dataset.upscale_inputs,
        static_fields_files=config_loader.dataset.static_fields_files,
        interpolation_mode=config_loader.dataset.interpolation_mode,
        coordinate_names=config_loader.dataset.coordinate_names,
        provide_coordinates=config_loader.dataset.provide_coordinates,
        provide_day_of_year=config_loader.dataset.provide_day_of_year
    ),
    batch_size=4,
    shuffle=True,
    num_workers=1,
)

input_files: ['standardized.q_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.q_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.q_850_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.t_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.t_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.t_850_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.u_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.u_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.u_850_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.v_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.v_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.v_850_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.z_500_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.z_700_CNRM-CM5_1961-1980_2080-2099.nc', 'standardized.z_850_CNRM-CM5_1961-1980_2080-2099.nc']
Reading static fields from ['scaled.orog_ALPS.nc']...
Providing ground truth coordinate grid...
Providing day of year encodings...


In [39]:
test_batch['LQ'].shape

torch.Size([4, 20, 128, 128])

In [40]:
test_batch['LQ'].isnan().any()

tensor(False)

In [41]:
test_batch['HQ'].shape

torch.Size([4, 2, 128, 128])

In [42]:
test_batch['HQ'].isnan().any()

tensor(False)